# TMS FACS scDRS+ heatmaps

Cleaned notebook for building marginal/conditional discovery tables, plotting curated/all-trait heatmaps, and producing Nature Genetics manuscript supplementary outputs.

The notebook writes one tidy cell-type-proportion CSV per heatmap and filters each per-trait `indep_cells` file using the same strict heatmap-section inclusion rule: the marginal × conditional proportion for that trait and cell type must be greater than the configured threshold (default: 5%). Placeholder signal labels such as `-1` are retained when their heatmap section passes.


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from anndata import AnnData, read_h5ad
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from statsmodels.stats.multitest import multipletests

# Keep notebook edits live while iterating on helper code.
%load_ext autoreload
%autoreload 2

## Inputs and trait sets

In [2]:
# === scDRS-FM reproduction: portable path anchor (injected, P5) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    # 1) explicit override wins
    env = _os.environ.get('SCDRSFM_BASE')
    if env:
        return _Path(env)
    # 2) search upward from CWD for the reproduction repo root
    #    (a directory containing both 'scDRS-FM-main' and 'scripts')
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'scDRS-FM-main').is_dir() and (d / 'scripts').is_dir():
            return d
    # 3) last resort: current working directory
    return here
BASE = _find_repo_root()
DATA = BASE / 'data'
RESULTS = BASE / 'results'
MAGMA_REF = BASE / 'magma_ref'
assert BASE.exists(), f'Repro root not found (set SCDRSFM_BASE to the repo root): {BASE}'


In [3]:
# Input data and scDRS+ result locations.
H5AD_FILE = str(DATA / "subsets_10k" / "TMS_FACS" / "TMS_FACS.h5ad")
GS_DIR = DATA / "gene_sets" / "gs_split"
RESULTS_DIR = RESULTS / "real" / "tms_facs"
INDEP_CELLS_DIR = Path("indep_cells/tms_facs")

# Nature Genetics manuscript supplementary tables generated by the heatmap cells.
MANUSCRIPT_SUPPLEMENTARY_DIR = Path("nature_genetics_manuscript_supplementary")
CURATED_HEATMAP_PROPORTIONS_CSV = (
    MANUSCRIPT_SUPPLEMENTARY_DIR / "TMS_FACS_curated_heatmap_cell_type_proportions.csv"
)
ALL_TRAIT_HEATMAP_PROPORTIONS_CSV = (
    MANUSCRIPT_SUPPLEMENTARY_DIR / "TMS_FACS_all_trait_heatmap_cell_type_proportions.csv"
)

# One source of truth for heatmap display, supplementary-table flags, and
# independent-cell filtering. Values must be strictly greater than this cutoff.
HEATMAP_THRESHOLD = 0.05

# Discover all trait gene-set files. Hidden files are ignored.
TRAITS: List[str] = sorted(
    p.name for p in GS_DIR.iterdir() if p.is_file() and not p.name.startswith(".")
)

# By default the build step runs all traits, matching the original notebook.
# Switch RUN_TRAITS to subset_traits below to run only the curated figure subset.
RUN_TRAITS = TRAITS


In [4]:
subset_traits = [
    'PASS_BIP_Mullins2021',
    'PASS_Schizophrenia_Pardinas2018',
    'PASS_MDD_Howard2019',
    #'PASS_Insomnia_Jansen2019',
    #'PASS_Intelligence_SavageJansen2018',
    'UKB_460K.body_BMIz',
    #'PASS_Type_1_Diabetes',
    'PASS_Rheumatoid_Arthritis',
    #'UKB_460K.disease_ASTHMA_DIAGNOSED',
    'PASS_IBD_deLange2017',
    'PASS_Multiple_sclerosis',
    'PASS_Lupus',
    'UKB_460K.blood_RBC_DISTRIB_WIDTH',
    'UKB_460K.biochemistry_Glucose',
    'PASS_Type_2_Diabetes',
    'PASS_AtrialFibrillation_Nielsen2018',
    'UKB_460K.bp_SYSTOLICadjMEDz',
    'UKB_460K.biochemistry_Cholesterol',
    'UKB_460K.biochemistry_LDLdirect',
    'UKB_460K.biochemistry_TotalProtein'
]


## Load and preprocess data

In [5]:
# Load and normalize the Tabula Muris Senis FACS data.
adata = read_h5ad(H5AD_FILE)
print(f"Loaded {H5AD_FILE}: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# Basic single-cell filtering / normalization used by the downstream discovery summaries.
sc.pp.filter_cells(adata, min_genes=250)
sc.pp.filter_genes(adata, min_cells=50)
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
print(f"After filtering: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

Loaded /mnt/shared-workspace/scdrsfm/data/subsets_10k/TMS_FACS/TMS_FACS.h5ad: 10,000 cells × 22,966 genes


After filtering: 10,000 cells × 16,722 genes


## Build trait × cell-type summaries

In [6]:
# -----------------------------------------------------------------------------
# Discovery summary helpers
# -----------------------------------------------------------------------------
def _pick_first_existing_col(
    df: pd.DataFrame,
    candidates: tuple[str, ...],
    *,
    what: str,
) -> str:
    """Return the first candidate column present in df, otherwise fail clearly."""
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"{what}: none of these columns exist: {candidates}")


def bh_fdr_mask(pvals: np.ndarray, alpha: float) -> np.ndarray:
    """Benjamini-Hochberg FDR mask, treating non-finite p-values as not significant."""
    pvals = np.asarray(pvals, dtype=np.float64)
    finite = np.isfinite(pvals)
    keep = np.zeros_like(finite, dtype=bool)

    if finite.sum() == 0:
        return keep

    rejected, _, _, _ = multipletests(pvals[finite], alpha=alpha, method="fdr_bh")
    keep[finite] = rejected
    return keep


def _unique_sorted_index(values: pd.Index | list[str]) -> pd.Index:
    """Return unique string values in stable sorted order where possible."""
    idx = pd.Index(values, dtype=str).unique()
    try:
        return idx.sort_values()
    except Exception:
        return idx


def _cells_from_metacells(df: pd.DataFrame, *, cell_ids_col: str = "cell_ids") -> pd.Index:
    """Expand comma-separated cell_ids from metacell rows into a unique cell-id index."""
    if cell_ids_col not in df.columns:
        raise ValueError(f"Conditional file is missing '{cell_ids_col}' column.")

    cells: list[str] = []
    for cell_ids in df[cell_ids_col].astype(str):
        if not cell_ids or cell_ids == "nan":
            continue
        cells.extend(cell_id for cell_id in cell_ids.split(",") if cell_id)

    return _unique_sorted_index(cells)


def _join_index(idx: pd.Index) -> str:
    """Serialize an index of IDs as a reproducible comma-separated string."""
    if idx is None or len(idx) == 0:
        return ""
    return ",".join(_unique_sorted_index(idx.astype(str)).tolist())


def _discovery_fraction_by_celltype(
    adata: AnnData,
    discovered_cell_ids: pd.Index,
    biocol: str,
    celltype_order: List[str],
    totals_by_type: pd.Series,
) -> pd.Series:
    """Fraction of cells discovered within each cell type."""
    discovered_cell_ids = adata.obs_names.intersection(discovered_cell_ids.astype(str))
    if len(discovered_cell_ids) == 0:
        return pd.Series(0.0, index=celltype_order)

    discovered_counts = adata.obs.loc[discovered_cell_ids, biocol].astype(str).value_counts()
    fractions = (discovered_counts / totals_by_type).reindex(celltype_order, fill_value=0.0)
    return fractions.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)


def _ctp_strings_by_celltype(
    adata: AnnData,
    cell_ids: pd.Index,
    biocol: str,
    celltype_order: List[str],
    totals_by_type: pd.Series,
) -> pd.Series:
    """Return per-cell-type 'causal/total' strings for a set of discovered cells."""
    cell_ids = adata.obs_names.intersection(cell_ids.astype(str))
    counts = (
        adata.obs.loc[cell_ids, biocol].astype(str).value_counts()
        if len(cell_ids)
        else pd.Series(dtype=int)
    )

    return pd.Series(
        {
            cell_type: f"{int(counts.get(cell_type, 0))}/{int(totals_by_type.get(cell_type, 0))}"
            for cell_type in celltype_order
        },
        index=celltype_order,
    )


def _valid_signal_values(series: pd.Series) -> List[int]:
    """Extract valid independent-signal IDs, excluding NaN and negative labels."""
    values = pd.to_numeric(series, errors="coerce")
    values = values[values.notna() & (values >= 0)]
    return sorted(values.astype(int).unique().tolist())


def _format_signal_value(value: float | int) -> int | float:
    """Keep integer-like signal values as ints for cleaner output files."""
    value_float = float(value)
    return int(value_float) if value_float.is_integer() else value_float


def _passes_heatmap_inclusion(value: float | int, threshold: float) -> bool:
    """Return True only when a finite proportion is strictly above the heatmap cutoff."""
    try:
        value_float = float(value)
        threshold_float = float(threshold)
    except (TypeError, ValueError):
        return False

    return bool(np.isfinite(value_float) and value_float > threshold_float)


def _marginal_x_conditional_cell_signal_assignments(
    *,
    adata: AnnData,
    marginal_sig_cells: pd.Index,
    df_cond_sig: pd.DataFrame,
    indep_sig_col: str,
) -> pd.DataFrame:
    """
    Build the requested per-trait table of marginal × conditional cell IDs and
    their associated independent-signal label.

    The table is generated from conditionally significant metacells only. It
    includes all non-null signal labels present in those rows, including -1 if
    the input uses -1 for a no-independent-signal label.
    """
    output_columns = ["marginal_x_conditional_cell_id", "independent_signal"]
    if len(df_cond_sig) == 0:
        return pd.DataFrame(columns=output_columns)

    signal_values = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce")
    signal_values = signal_values[signal_values.notna()]
    if len(signal_values) == 0:
        return pd.DataFrame(columns=output_columns)

    rows: list[pd.DataFrame] = []
    for signal_value in sorted(signal_values.unique()):
        signal_mask = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce").eq(signal_value).fillna(False)
        signal_metacells = df_cond_sig.loc[signal_mask]
        signal_cells = adata.obs_names.intersection(_cells_from_metacells(signal_metacells))
        marginal_x_conditional_cells = marginal_sig_cells.intersection(signal_cells)

        if len(marginal_x_conditional_cells) == 0:
            continue

        rows.append(
            pd.DataFrame(
                {
                    "marginal_x_conditional_cell_id": _unique_sorted_index(marginal_x_conditional_cells),
                    "independent_signal": _format_signal_value(signal_value),
                }
            )
        )

    if not rows:
        return pd.DataFrame(columns=output_columns)

    assignments = pd.concat(rows, ignore_index=True)
    assignments = assignments.drop_duplicates(output_columns)
    assignments = assignments.sort_values(
        ["independent_signal", "marginal_x_conditional_cell_id"],
        kind="mergesort",
    ).reset_index(drop=True)
    return assignments


def _filter_independent_cell_assignments_for_heatmap(
    *,
    adata: AnnData,
    assignments: pd.DataFrame,
    heatmap_cell_ids: pd.Index,
    biocol: str,
    totals_by_type: pd.Series,
    threshold: float,
) -> pd.DataFrame:
    """
    Keep assignments only from trait × cell-type sections included in the heatmap.

    Section inclusion is calculated from all marginal × conditional cells for the
    trait, using the same cell-type denominator and strict ``> threshold`` rule as
    the heatmap. Every assignment in a passing section is retained, including
    placeholder labels such as -1, so traits without a nonnegative independent
    signal remain consistent with the heatmap's fallback population annotation.
    """
    output_columns = ["marginal_x_conditional_cell_id", "independent_signal"]
    if assignments is None or len(assignments) == 0:
        return pd.DataFrame(columns=output_columns)

    missing_columns = [column for column in output_columns if column not in assignments.columns]
    if missing_columns:
        raise ValueError(f"Independent-cell assignments are missing columns: {missing_columns}")
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs is missing '{biocol}'")

    celltype_order = totals_by_type.index.astype(str).tolist()
    section_fractions = _discovery_fraction_by_celltype(
        adata=adata,
        discovered_cell_ids=heatmap_cell_ids,
        biocol=biocol,
        celltype_order=celltype_order,
        totals_by_type=totals_by_type,
    )
    passing_cell_types = set(
        section_fractions.index[
            section_fractions.map(
                lambda value: _passes_heatmap_inclusion(value, threshold)
            )
        ].astype(str)
    )
    if not passing_cell_types:
        return pd.DataFrame(columns=output_columns)

    work = assignments.loc[:, output_columns].copy()
    work["marginal_x_conditional_cell_id"] = work[
        "marginal_x_conditional_cell_id"
    ].astype(str)

    valid_cell_ids = adata.obs_names.intersection(
        work["marginal_x_conditional_cell_id"].astype(str)
    )
    work = work.loc[
        work["marginal_x_conditional_cell_id"].isin(valid_cell_ids)
    ].copy()
    if len(work) == 0:
        return pd.DataFrame(columns=output_columns)

    cell_type_by_id = adata.obs[biocol].astype(str)
    assignment_cell_types = work["marginal_x_conditional_cell_id"].map(cell_type_by_id)
    work = work.loc[assignment_cell_types.isin(passing_cell_types), output_columns]
    work = work.drop_duplicates(output_columns)
    work = work.sort_values(
        ["independent_signal", "marginal_x_conditional_cell_id"],
        kind="mergesort",
    ).reset_index(drop=True)
    return work


# -----------------------------------------------------------------------------
# Main build function
# -----------------------------------------------------------------------------
def build_trait_by_celltype_proportions_dfs(
    *,
    adata: AnnData,
    out_folder: Path,
    traits: List[str],
    biocol: str = "cluster.id",
    marginal_metacell_col: str = "metacell",
    fdr_alpha: float = 0.1,
    heatmap_threshold: float = 0.05,
    pval_col_candidates: tuple[str, ...] = ("pval", "mc_pval"),
    print_indep_signal_summaries: bool = True,
    indep_sig_col: str = "independent_signal",
    indep_cells_dir: Path | str = Path("indep_cells/tms_facs"),
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build trait × cell-type discovery summaries and save heatmap-filtered independent-cell files.

    Returns
    -------
    df_marginal
        Fraction of each cell type discovered by marginal scores.
    df_intersect
        Fraction of each cell type discovered by marginal × conditional scores.
    df_indep_signal_counts
        Per-trait independent-signal counts.
    df_signal_ctp
        Per-(trait, independent_signal) 'causal/total' strings by cell type.
    df_signal_details
        Per-(trait, independent_signal) metacell and cell-id details.

    Side effect
    -----------
    Writes one gzip-compressed TSV per trait to
    ``indep_cells_dir / f"{trait}.gz"`` with columns
    ``marginal_x_conditional_cell_id`` and ``independent_signal``. Only
    assignments from trait × cell-type heatmap sections whose marginal ×
    conditional proportion is strictly greater than ``heatmap_threshold`` are
    written. Signal labels are otherwise preserved, including -1 placeholders.
    """
    out_folder = Path(out_folder)
    indep_cells_dir = Path(indep_cells_dir)
    indep_cells_dir.mkdir(parents=True, exist_ok=True)

    heatmap_threshold = float(heatmap_threshold)
    if not np.isfinite(heatmap_threshold) or not 0 <= heatmap_threshold <= 1:
        raise ValueError("heatmap_threshold must be a finite value between 0 and 1.")

    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs is missing '{biocol}'")

    # Stable column order and denominators for discovery fractions.
    celltype_order = adata.obs[biocol].astype(str).value_counts().index.tolist()
    totals_by_type = adata.obs[biocol].astype(str).value_counts().reindex(celltype_order)

    rows_marginal: Dict[str, pd.Series] = {}
    rows_intersect: Dict[str, pd.Series] = {}
    trait_counts_rows: Dict[str, Dict[str, int]] = {}
    signal_ctp_rows: Dict[Tuple[str, int], pd.Series] = {}
    signal_details_rows: Dict[Tuple[str, int], Dict[str, object]] = {}

    for trait in traits:
        prefix = Path(trait).name
        marginal_file = out_folder / f"{prefix}.marginal_score.gz"
        cond_file = out_folder / f"{prefix}.conditional.tagging_score.gz"

        # Marginal significant cells.
        df_marginal_score = pd.read_csv(marginal_file, sep="\t", compression="gzip", index_col=0)
        if marginal_metacell_col not in df_marginal_score.columns:
            raise ValueError(f"Marginal file for trait '{trait}' is missing '{marginal_metacell_col}'.")

        marginal_pval_col = _pick_first_existing_col(
            df_marginal_score,
            tuple(pval_col_candidates),
            what="marginal",
        )
        marginal_sig_mask = bh_fdr_mask(df_marginal_score[marginal_pval_col].to_numpy(), fdr_alpha)
        marginal_sig_cells = adata.obs_names.intersection(df_marginal_score.index[marginal_sig_mask].astype(str))

        # Conditional metacell scores.
        df_cond = pd.read_csv(cond_file, sep="\t", compression="gzip", index_col=0).copy()
        df_cond.index = pd.to_numeric(pd.Index(df_cond.index), errors="coerce")
        df_cond = df_cond.loc[df_cond.index.notna()]
        df_cond.index = df_cond.index.astype(int)

        if indep_sig_col not in df_cond.columns:
            raise ValueError(f"Conditional file for trait '{trait}' is missing '{indep_sig_col}'.")
        if "cell_ids" not in df_cond.columns:
            raise ValueError(f"Conditional file for trait '{trait}' is missing 'cell_ids'.")

        conditional_pval_col = _pick_first_existing_col(
            df_cond,
            tuple(pval_col_candidates),
            what="conditional",
        )
        conditional_sig_mask = bh_fdr_mask(df_cond[conditional_pval_col].to_numpy(), fdr_alpha)
        df_cond_sig = df_cond.loc[conditional_sig_mask] if conditional_sig_mask.any() else df_cond.iloc[0:0]

        cells_in_cond_sig_metacells = (
            adata.obs_names.intersection(_cells_from_metacells(df_cond_sig))
            if len(df_cond_sig)
            else pd.Index([], dtype=str)
        )
        intersect_cells = marginal_sig_cells.intersection(cells_in_cond_sig_metacells)

        # Per-trait output: keep only trait × cell-type sections that pass the
        # same strict marginal × conditional proportion threshold as the heatmap.
        independent_cell_assignments_unfiltered = _marginal_x_conditional_cell_signal_assignments(
            adata=adata,
            marginal_sig_cells=marginal_sig_cells,
            df_cond_sig=df_cond_sig,
            indep_sig_col=indep_sig_col,
        )
        independent_cell_assignments = _filter_independent_cell_assignments_for_heatmap(
            adata=adata,
            assignments=independent_cell_assignments_unfiltered,
            heatmap_cell_ids=intersect_cells,
            biocol=biocol,
            totals_by_type=totals_by_type,
            threshold=heatmap_threshold,
        )
        independent_cell_assignments.to_csv(
            indep_cells_dir / f"{prefix}.gz",
            sep="\t",
            index=False,
            compression="gzip",
        )

        # Trait-level discovery fractions.
        frac_marginal = _discovery_fraction_by_celltype(
            adata,
            marginal_sig_cells,
            biocol,
            celltype_order,
            totals_by_type,
        )
        frac_marginal.name = trait
        rows_marginal[trait] = frac_marginal

        frac_intersect = _discovery_fraction_by_celltype(
            adata,
            intersect_cells,
            biocol,
            celltype_order,
            totals_by_type,
        )
        frac_intersect.name = trait
        rows_intersect[trait] = frac_intersect

        # Independent-signal counts and per-signal summaries.
        signal_labels_all = pd.to_numeric(df_cond[indep_sig_col], errors="coerce")
        signal_labels_cond_sig = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce") if len(df_cond_sig) else pd.Series(dtype=float)
        valid_signals_total = _valid_signal_values(df_cond[indep_sig_col])
        valid_signals_cond_sig = _valid_signal_values(df_cond_sig[indep_sig_col]) if len(df_cond_sig) else []

        n_signals_with_causal_cells = 0
        for signal_id in valid_signals_total:
            # All metacells assigned to this signal, regardless of conditional significance.
            all_signal_mask = signal_labels_all.astype("Int64").eq(signal_id).fillna(False)
            signal_rows_all = df_cond.loc[all_signal_mask]
            metacells_all = pd.Index(signal_rows_all.index.astype(int)).unique()
            cells_all = (
                adata.obs_names.intersection(_cells_from_metacells(signal_rows_all))
                if len(signal_rows_all)
                else pd.Index([], dtype=str)
            )
            marginal_x_signal_all = marginal_sig_cells.intersection(cells_all)

            # Conditionally significant metacells assigned to this signal.
            if len(df_cond_sig):
                cond_sig_signal_mask = signal_labels_cond_sig.astype("Int64").eq(signal_id).fillna(False)
                signal_rows_cond_sig = df_cond_sig.loc[cond_sig_signal_mask]
            else:
                signal_rows_cond_sig = df_cond_sig

            metacells_cond_sig = (
                pd.Index(signal_rows_cond_sig.index.astype(int)).unique()
                if len(signal_rows_cond_sig)
                else pd.Index([], dtype=int)
            )
            cells_cond_sig = (
                adata.obs_names.intersection(_cells_from_metacells(signal_rows_cond_sig))
                if len(signal_rows_cond_sig)
                else pd.Index([], dtype=str)
            )
            marginal_x_signal_cond_sig = marginal_sig_cells.intersection(cells_cond_sig)

            signal_details_rows[(trait, signal_id)] = {
                "n_metacells_in_signal_all": int(len(metacells_all)),
                "metacell_ids_in_signal_all": _join_index(metacells_all.astype(str)),
                "n_cells_in_signal_all": int(len(cells_all)),
                "cell_ids_in_signal_all": _join_index(cells_all),
                "n_marg_x_signal_cells_all": int(len(marginal_x_signal_all)),
                "marg_x_signal_cell_ids_all": _join_index(marginal_x_signal_all),
                "n_metacells_in_signal_cond_sig": int(len(metacells_cond_sig)),
                "metacell_ids_in_signal_cond_sig": _join_index(metacells_cond_sig.astype(str)),
                "n_cells_in_signal_cond_sig": int(len(cells_cond_sig)),
                "cell_ids_in_signal_cond_sig": _join_index(cells_cond_sig),
                "n_marg_x_signal_cells_cond_sig": int(len(marginal_x_signal_cond_sig)),
                "marg_x_signal_cell_ids_cond_sig": _join_index(marginal_x_signal_cond_sig),
            }

            if len(marginal_x_signal_cond_sig) > 0:
                n_signals_with_causal_cells += 1
                ctp = _ctp_strings_by_celltype(
                    adata=adata,
                    cell_ids=marginal_x_signal_cond_sig,
                    biocol=biocol,
                    celltype_order=celltype_order,
                    totals_by_type=totals_by_type,
                )
                ctp.name = (trait, signal_id)
                signal_ctp_rows[(trait, signal_id)] = ctp

        trait_counts_rows[trait] = {
            "n_independent_signals_total": int(len(valid_signals_total)),
            "n_independent_signals_cond_sig": int(len(valid_signals_cond_sig)),
            "n_independent_signals_with_any_causal_cells": int(n_signals_with_causal_cells),
            "n_marginal_sig_cells": int(len(marginal_sig_cells)),
            "n_cond_sig_cells": int(len(cells_in_cond_sig_metacells)),
            "n_causal_cells_marg_x_cond": int(len(intersect_cells)),
            "n_indep_cell_assignments_before_heatmap_filter": int(
                len(independent_cell_assignments_unfiltered)
            ),
            "n_indep_cell_assignments_saved_after_heatmap_filter": int(
                len(independent_cell_assignments)
            ),
        }

        if print_indep_signal_summaries:
            print(
                f"[{trait}] independent signals: total={len(valid_signals_total)}, "
                f"cond-sig={len(valid_signals_cond_sig)}, "
                f"with_any_causal_cells={n_signals_with_causal_cells}; "
                f"causal_cells(marg∩cond)={len(intersect_cells)}; "
                f"indep_assignments(raw={len(independent_cell_assignments_unfiltered)}, "
                f"saved_in_sections_>{heatmap_threshold:.1%}={len(independent_cell_assignments)}); "
                f"saved={indep_cells_dir / f'{prefix}.gz'}"
            )
            if n_signals_with_causal_cells > 0:
                trait_signal_ctp = pd.DataFrame.from_dict(
                    {sig: series for (t, sig), series in signal_ctp_rows.items() if t == trait},
                    orient="index",
                ).reindex(columns=celltype_order)
                trait_signal_ctp.index.name = "independent_signal"
                trait_signal_ctp.columns.name = biocol
                display(trait_signal_ctp)

    df_marginal = pd.DataFrame.from_dict(rows_marginal, orient="index").astype(float)
    df_marginal.index.name = "trait"
    df_marginal.columns.name = biocol

    df_intersect = pd.DataFrame.from_dict(rows_intersect, orient="index").astype(float)
    df_intersect.index.name = "trait"
    df_intersect.columns.name = biocol

    df_indep_signal_counts = pd.DataFrame.from_dict(trait_counts_rows, orient="index").astype(int)
    df_indep_signal_counts.index.name = "trait"

    if signal_ctp_rows:
        df_signal_ctp = pd.DataFrame.from_dict(signal_ctp_rows, orient="index").reindex(columns=celltype_order)
        df_signal_ctp.index = pd.MultiIndex.from_tuples(
            df_signal_ctp.index,
            names=["trait", "independent_signal"],
        )
        df_signal_ctp.columns.name = biocol
    else:
        df_signal_ctp = pd.DataFrame(columns=celltype_order)
        df_signal_ctp.index = pd.MultiIndex.from_tuples([], names=["trait", "independent_signal"])
        df_signal_ctp.columns.name = biocol

    signal_detail_columns = [
        "n_metacells_in_signal_all",
        "metacell_ids_in_signal_all",
        "n_cells_in_signal_all",
        "cell_ids_in_signal_all",
        "n_marg_x_signal_cells_all",
        "marg_x_signal_cell_ids_all",
        "n_metacells_in_signal_cond_sig",
        "metacell_ids_in_signal_cond_sig",
        "n_cells_in_signal_cond_sig",
        "cell_ids_in_signal_cond_sig",
        "n_marg_x_signal_cells_cond_sig",
        "marg_x_signal_cell_ids_cond_sig",
    ]
    if signal_details_rows:
        df_signal_details = pd.DataFrame.from_dict(signal_details_rows, orient="index")
        df_signal_details = df_signal_details.reindex(columns=signal_detail_columns)
        df_signal_details.index = pd.MultiIndex.from_tuples(
            df_signal_details.index,
            names=["trait", "independent_signal"],
        )
    else:
        df_signal_details = pd.DataFrame(columns=signal_detail_columns)
        df_signal_details.index = pd.MultiIndex.from_tuples([], names=["trait", "independent_signal"])

    return df_marginal, df_intersect, df_indep_signal_counts, df_signal_ctp, df_signal_details

In [7]:
(
    df_marginal_props,
    df_marginal_x_cond_props,
    df_indep_signal_counts,
    df_signal_ctp,
    df_signal_details,
) = build_trait_by_celltype_proportions_dfs(
    adata=adata,
    out_folder=RESULTS_DIR,
    traits=RUN_TRAITS,
    biocol="cell_ontology_class",
    marginal_metacell_col="metacell",
    fdr_alpha=0.1,
    heatmap_threshold=HEATMAP_THRESHOLD,
    pval_col_candidates=("pval",),
    print_indep_signal_summaries=True,
    indep_sig_col="independent_signal_multi",
    indep_cells_dir=INDEP_CELLS_DIR,
)

print("Marginal-only cell-type discovery fractions (discovered / total-in-celltype):")
display(df_marginal_props)

print("Marginal × conditional-metacell-cell discovery fractions (discovered / total-in-celltype):")
display(df_marginal_x_cond_props)

print("Independent-signal counts per trait:")
display(df_indep_signal_counts)

print("Per-(trait, independent_signal) cell-type '{causal}/{total}' strings:")
display(df_signal_ctp)

print("Per-(trait, independent_signal) detailed signal info:")
display(df_signal_details)

[PASS_ADHD_Demontis2018] independent signals: total=0, cond-sig=0, with_any_causal_cells=0; causal_cells(marg∩cond)=0; indep_assignments(raw=0, saved_in_sections_>5.0%=0); saved=indep_cells/tms_facs/PASS_ADHD_Demontis2018.gz


[PASS_Alzheimers_Jansen2019] independent signals: total=0, cond-sig=0, with_any_causal_cells=0; causal_cells(marg∩cond)=0; indep_assignments(raw=0, saved_in_sections_>5.0%=0); saved=indep_cells/tms_facs/PASS_Alzheimers_Jansen2019.gz


[PASS_AtrialFibrillation_Nielsen2018] independent signals: total=7, cond-sig=2, with_any_causal_cells=2; causal_cells(marg∩cond)=50; indep_assignments(raw=50, saved_in_sections_>5.0%=41); saved=indep_cells/tms_facs/PASS_AtrialFibrillation_Nielsen2018.gz


cell_ontology_class,microglial cell,B cell,endothelial cell,mesenchymal stem cell of adipose,basal cell of epidermis,fibroblast of cardiac tissue,hematopoietic stem cell,granulocyte,bulge keratinocyte,naive B cell,...,keratinocyte stem cell,respiratory basal cell,fibrocyte,lung macrophage,ciliated columnar cell of tracheobronchial tree,club cell of bronchiole,plasmacytoid dendritic cell,plasma cell,kidney interstitial fibroblast,lymphocyte
independent_signal,,,,,,,,,,,,,,,,,,,,,
18,0/1196,0/555,0/444,0/437,0/396,0/354,0/285,0/268,0/261,0/255,...,0/3,0/3,0/3,0/2,0/2,0/2,0/1,0/1,0/1,0/1
227,0/1196,0/555,0/444,0/437,0/396,0/354,0/285,0/268,0/261,0/255,...,0/3,0/3,0/3,0/2,0/2,0/2,0/1,0/1,0/1,0/1


[PASS_BIP_Mullins2021] independent signals: total=6, cond-sig=2, with_any_causal_cells=2; causal_cells(marg∩cond)=144; indep_assignments(raw=144, saved_in_sections_>5.0%=144); saved=indep_cells/tms_facs/PASS_BIP_Mullins2021.gz


...

,n_independent_signals_total,n_independent_signals_cond_sig,n_independent_signals_with_any_causal_cells,n_marginal_sig_cells,n_cond_sig_cells,n_causal_cells_marg_x_cond,n_indep_cell_assignments_before_heatmap_filter,n_indep_cell_assignments_saved_after_heatmap_filter
trait,,,,,,,,
PASS_ADHD_Demontis2018,0,0,0,62,0,0,0,0
PASS_Alzheimers_Jansen2019,0,0,0,0,0,0,0,0
PASS_AtrialFibrillation_Nielsen2018,7,2,2,55,59,50,50,41
PASS_BIP_Mullins2021,6,2,2,223,144,144,144,144
PASS_CD_deLange2017,0,0,0,1990,497,476,476,460
...,...,...,...,...,...,...,...,...
UKB_460K.other_MORNINGPERSON,6,4,4,321,201,157,157,138
UKB_460K.pigment_HAIR,2,1,1,733,189,189,189,189
UKB_460K.repro_MENARCHE_AGE,0,0,0,82,67,65,65,64


Per-(trait, independent_signal) cell-type '{causal}/{total}' strings:


cell_ontology_class                                    microglial cell  \
trait                               independent_signal                   
PASS_AtrialFibrillation_Nielsen2018 18                          0/1196   
                                    227                         0/1196   
PASS_BIP_Mullins2021                23                          0/1196   
                                    38                          0/1196   
PASS_FastingGlucose_Manning         46                          0/1196   
...                                                                ...   
UKB_460K.other_MORNINGPERSON        38                          0/1196   
                                    46                          0/1196   
                                    131                         0/1196   
UKB_460K.pigment_HAIR               185                         0/1196   
UKB_460K.repro_MENOPAUSE_AGE        669                         0/1196   

cell_ontology_class                                     B cell  \
trait                               independent_signal           
PASS_AtrialFibrillation_Nielsen2018 18                   0/555   
                                    227                  0/555   
PASS_BIP_Mullins2021                23                   0/555   
                                    38                   0/555   
PASS_FastingGlucose_Manning         46                   0/555   
...                                                        ...   
UKB_460K.other_MORNINGPERSON        38                   0/555   
                                    46                   0/555   
                                    131                  0/555   
UKB_460K.pigment_HAIR               185                  0/555   
UKB_460K.repro_MENOPAUSE_AGE        669                 95/555   

cell_ontology_class                                    endothelial cell  \
trait                               independent_signal                    
PASS_AtrialFibrillation_Nielsen2018 18                            0/444   
                                    227                           0/444   
PASS_BIP_Mullins2021                23                            0/444   
                                    38                            0/444   
PASS_FastingGlucose_Manning         46                            0/444   
...                                                                 ...   
UKB_460K.other_MORNINGPERSON        38                            0/444   
                                    46                            0/444   
                                    131                           0/444   
UKB_460K.pigment_HAIR               185                           0/444   
UKB_460K.repro_MENOPAUSE_AGE        669                           0/444   

cell_ontology_class                                    mesenchymal stem cell of adipose  \
trait                               independent_signal                                    
PASS_AtrialFibrillation_Nielsen2018 18                                            0/437   
                                    227                                           0/437   
PASS_BIP_Mullins2021                23                                            0/437   
                                    38                                            0/437   
PASS_FastingGlucose_Manning         46                                            0/437   
...                                                                                 ...   
UKB_460K.other_MORNINGPERSON        38                                            0/437   
                                    46                                            0/437   
                                    131                                           0/437   
UKB_460K.pigment_HAIR               185                                           0/437   
UKB_460K.repro_MENOPAUSE_AGE        669                                           0/437   

cell_ontology_clas

Per-(trait, independent_signal) detailed signal info:


n_metacells_in_signal_all  \
trait                               independent_signal                              
PASS_AtrialFibrillation_Nielsen2018 18                                          4   
                                    117                                         1   
                                    159                                        13   
                                    227                                       493   
                                    335                                       150   
...                                                                           ...   
UKB_460K.other_MORNINGPERSON        162                                         9   
UKB_460K.pigment_HAIR               64                                         13   
                                    185                                       916   
UKB_460K.repro_MENOPAUSE_AGE        43                                         30   
                                    669                                       899   

                                                                               metacell_ids_in_signal_all  \
trait                               independent_signal                                                      
PASS_AtrialFibrillation_Nielsen2018 18                                                     18,287,344,847   
                                    117                                                               117   
                                    159                   106,11,159,212,32,41,435,482,486,558,563,59,705   
                                    227                 0,1,100,101,104,105,107,109,110,111,113,114,11...   
                                    335                 10,102,112,120,124,125,127,131,146,151,152,157...   
...                                                                                                   ...   
UKB_460K.other_MORNINGPERSON        162                                 162,213,23,25,278,281,287,355,378   
UKB_460K.pigment_HAIR               64                     131,253,277,326,37,39,458,474,515,60,620,63,64   
                                    185                 0,1,10,100,101,102,103,104,105,106,107,108,109...   
UKB_460K.repro_MENOPAUSE_AGE        43                  124,126,133,154,26,264,272,28,283,300,325,327,...   
                                    669                 0,1,10,100,101,102,103,104,105,106,107,108,109...   

                                                        n_cells_in_signal_all  \
trait                               independent_signal                          
PASS_AtrialFibrillation_Nielsen2018 18                                     51   
                                    117                                    16   
                                    159                                   178   
                                    227                                  4785   
                                    335                                  1680   
...                                                                       ...   
UKB_460K.other_MORNINGPERSON        162                                   128   
UKB_460K.pigment_HAIR               64                                    179   
                                    185                                  9334   
UKB_460K.repro_MENOPAUSE_AGE        43                                    395   
                                    669                                  9118   

                                                                                   cell_ids_in_signal_all  \
trait                               independent_signal                                                      
PASS_AtrialFibrillation_Nielsen2018 18                  A5.MAA100037.3_10_M.1.1-1-1,A8_B008659_S68_L00...   
                                    117                 A19_B000491_B009016_S19.mm10-plus-8-0,D20_B000...   
                                    159   

## Curated subset labels

In [8]:
trait_dict = {
    "PASS_BIP_Mullins2021": "Bipolar Disorder (BIP)",
    "PASS_Schizophrenia_Pardinas2018": "Schizophrenia (SCZ)",
    "PASS_MDD_Howard2019": "Major Depressive Disorder (MDD)",
    "UKB_460K.mental_NEUROTICISM": "Neuroticism",
    "PASS_Intelligence_SavageJansen2018": "Intelligence (IQ)",
    "PASS_Insomnia_Jansen2019": "Insomnia",
    "UKB_460K.body_BMIz": "Body Mass Index (BMI)",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes (T1D)",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid Arthritis (RA)",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma",
    "PASS_IBD_deLange2017": "Inflammatory Bowel Disease (IBD)",
    "PASS_Multiple_sclerosis": "Multiple Sclerosis (MS)",
    "PASS_Lupus": "Lupus",
    "UKB_460K.blood_RBC_DISTRIB_WIDTH": "Red Blood Cell Distribution Width (RDW)",
    "UKB_460K.biochemistry_Glucose": "Glucose",
    "PASS_Type_2_Diabetes": "Type 2 Diabetes (T2D)",
    "PASS_AtrialFibrillation_Nielsen2018": "Atrial Fibrillation (AF)",
    "UKB_460K.bp_SYSTOLICadjMEDz": "Systolic Blood Pressure (SBP)",
    "UKB_460K.biochemistry_Cholesterol": "Total Cholesterol (TC)",
    "UKB_460K.biochemistry_LDLdirect": "Low-Density Lipoprotein Cholesterol (LDL)",
    "UKB_460K.biochemistry_TotalProtein": "Total Protein (TP)",
}

missing_subset_labels = [trait for trait in subset_traits if trait not in trait_dict]
if missing_subset_labels:
    raise ValueError(f"Missing subset trait labels: {missing_subset_labels}")

In [9]:
plot_cell_types = [
    # Neuronal
    "neuron",
    "interneuron",
    "medium spiny neuron",
    "oligodendrocyte precursor cell",
    "oligodendrocyte",
    "neuroepithelial cell",
    "astrocyte",
    "microglial cell",


    # Immune / blood
    "regulatory T cell",
    "CD4-positive, alpha-beta T cell",
    "CD8-positive, alpha-beta T cell",
    "NK cell",
    "naive B cell",
    "immature B cell",
    "precursor B cell",
    "dendritic cell",
    "granulocyte",
    "thymocyte",
    "promonocyte",

    # Other / metabolic / tissue
    "proerythroblast",
    "pancreatic B cell",  # PB
    "pancreatic A cell",  # PA
    "pancreatic ductal cell",
    "atrial myocyte",
    "ventricular myocyte",
    "pericyte cell",
    "hepatocyte",
    "secretory cell",
    "endothelial cell of hepatic sinusoid",
    "epithelial cell of thymus",
    "smooth muscle cell of trachea",
    "basal cell",
]
cell_type_dict = {
    "neuron": "Neuron (484)",
    "interneuron": "Interneuron (240)",
    "medium spiny neuron": "Medium spiny neuron (103)",
    "oligodendrocyte precursor cell": "Oligodendrocyte precursor cell (312)",
    "oligodendrocyte": "Oligodendrocyte (2094)",
    "neuroepithelial cell": "Neuroepithelial cell (151)",
    "astrocyte": "Astrocyte (592)",
    "microglial cell": "Microglial cell (13268)",

    "regulatory T cell": "Regulatory T cell (47)",
    "CD4-positive, alpha-beta T cell": "CD4-positive, alpha-beta T cell (1170)",
    "CD8-positive, alpha-beta T cell": "CD8-positive, alpha-beta T cell (1070)",
    "NK cell": "NK cell (877)",
    "dendritic cell": "Dendritic cell (62)",
    "naive B cell": "Naive B cell (2959)",
    "immature B cell": "Immature B cell (589)",

    "proerythroblast": "Proerythroblast (107)",
    "pancreatic B cell": "Pancreatic beta cell (1342)",
    "pancreatic A cell": "Pancreatic alpha cell (521)",
    "atrial myocyte": "Atrial myocyte (403)",
    "ventricular myocyte": "Ventricular myocyte (147)",
    "pericyte cell": "Pericyte (38)",
    "hepatocyte": "Hepatocyte (1162)",
    "secretory cell": "Secretory cell (563)",
    "endothelial cell of hepatic sinusoid": "Endothelial cell of hepatic sinusoid (617)",
}

## Shared plot helpers

In [10]:
# -----------------------------------------------------------------------------
# Plotting helpers shared by the curated and all-trait heatmaps
# -----------------------------------------------------------------------------
import re


def _capfirst(label: str) -> str:
    label = str(label).strip()
    return (label[0].upper() + label[1:]) if label else label


def _strip_trailing_count(label: str) -> str:
    """Remove a trailing ' (123)' count if present."""
    return re.sub(r"\s*\(\d+\)\s*$", "", str(label).strip())


def _display_cell_type_name(cell_type: str, cell_type_dict: dict[str, str] | None = None) -> str:
    cell_type_dict = cell_type_dict or {}
    return cell_type_dict.get(cell_type, cell_type)


def _cell_type_count_map(adata: AnnData | None, adata_biocol: str = "cell_ontology_class") -> pd.Series:
    """Return total cells per cell type, or an empty count series when adata is unavailable."""
    if adata is None or adata_biocol not in adata.obs.columns:
        return pd.Series(dtype=int)
    return adata.obs[adata_biocol].astype(str).value_counts()


def _cell_type_label_with_count(
    cell_type: str,
    *,
    adata: AnnData | None = None,
    adata_biocol: str = "cell_ontology_class",
    cell_type_dict: dict[str, str] | None = None,
) -> str:
    """Pretty cell-type label with an explicit total-cell count."""
    base_label = _strip_trailing_count(_display_cell_type_name(cell_type, cell_type_dict))
    base_label = _capfirst(base_label)

    counts = _cell_type_count_map(adata, adata_biocol)
    if len(counts) == 0:
        return base_label

    return f"{base_label} ({int(counts.get(str(cell_type), 0)):,})"


def _color_ticklabels(ticklabels, colors, *, fontsize=24, rotation=None, ha=None) -> None:
    for tick, color in zip(ticklabels, colors):
        tick.set_color(color)
        tick.set_fontsize(fontsize)
        if rotation is not None:
            tick.set_rotation(rotation)
        if ha is not None:
            tick.set_ha(ha)


def _ordered_with_remainder(observed_items, preferred_order):
    observed_items = [str(item) for item in observed_items]
    preferred_order = [str(item) for item in preferred_order]

    observed_set = set(observed_items)
    ordered = [item for item in preferred_order if item in observed_set]
    ordered_set = set(ordered)
    remainder = [item for item in observed_items if item not in ordered_set]
    return ordered + remainder


def _ordered_by_named_groups(
    observed_items: list[str],
    groups: dict[str, list[str]],
    *,
    fallback_group: str = "Other",
) -> tuple[list[str], dict[str, list[str]]]:
    """
    Order observed items by named groups while keeping the groups contiguous.

    Items missing from the group definitions are appended to fallback_group in
    their original observed order.
    """
    observed_items = [str(item) for item in observed_items]
    observed_set = set(observed_items)

    grouped: dict[str, list[str]] = {name: [] for name in groups}
    used: set[str] = set()

    for group_name, preferred_items in groups.items():
        for item in [str(x) for x in preferred_items]:
            if item in observed_set and item not in used:
                grouped[group_name].append(item)
                used.add(item)

    if fallback_group not in grouped:
        grouped[fallback_group] = []

    for item in observed_items:
        if item not in used:
            grouped[fallback_group].append(item)
            used.add(item)

    ordered_items: list[str] = []
    for group_name in groups:
        ordered_items.extend(grouped[group_name])

    return ordered_items, grouped


def _parse_cell_ids_csv(cell_ids: str) -> pd.Index:
    if cell_ids is None:
        return pd.Index([], dtype=str)

    cell_ids = str(cell_ids)
    if not cell_ids or cell_ids == "nan":
        return pd.Index([], dtype=str)

    return pd.Index([cell_id for cell_id in cell_ids.split(",") if cell_id], dtype=str)


def _build_indep_sig_annotations(
    *,
    adata: AnnData,
    df_signal_details: pd.DataFrame | None,
    adata_biocol: str,
    cell_types: list[str],
    threshold: float,
    trait_index: list[str],
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
) -> tuple[dict[str, dict[str, str]], set[str]]:
    """
    Map each trait/cell type to independent-signal annotations that exceed a fraction threshold.

    Returns
    -------
    ann_map
        ann_map[trait][cell_type] = "1,2,..." for threshold-passing signals.
    no_discovery_traits
        Traits with no marginal × conditional cells; these receive the fallback annotation.
    """
    ann_map: dict[str, dict[str, str]] = {trait: {cell_type: "" for cell_type in cell_types} for trait in trait_index}
    no_discovery_traits: set[str] = set()

    if df_signal_details is None or len(df_signal_details) == 0:
        no_discovery_traits.update(trait_index)
        return ann_map, no_discovery_traits

    if not isinstance(df_signal_details.index, pd.MultiIndex) or df_signal_details.index.nlevels != 2:
        raise ValueError("df_signal_details must have a MultiIndex with levels ['trait', 'independent_signal'].")

    if adata_biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs is missing '{adata_biocol}'")

    totals_by_type = adata.obs[adata_biocol].astype(str).value_counts()
    traits_available = set(df_signal_details.index.get_level_values(0).astype(str))

    for trait in trait_index:
        trait = str(trait)
        if trait not in traits_available:
            no_discovery_traits.add(trait)
            continue

        trait_signal_details = df_signal_details.xs(trait, level=0, drop_level=False)
        if signal_cell_ids_col not in trait_signal_details.columns:
            raise ValueError(f"df_signal_details is missing '{signal_cell_ids_col}'")

        signal_cell_counts = pd.to_numeric(
            trait_signal_details.get(
                "n_marg_x_signal_cells_cond_sig",
                pd.Series(index=trait_signal_details.index, data=np.nan),
            ),
            errors="coerce",
        )
        if signal_cell_counts.notna().any():
            keep_signal = signal_cell_counts.fillna(0).astype(float) > 0
        else:
            keep_signal = trait_signal_details[signal_cell_ids_col].astype(str).map(
                lambda value: len(_parse_cell_ids_csv(value)) > 0
            )

        trait_signal_details = trait_signal_details.loc[keep_signal]
        if len(trait_signal_details) == 0:
            no_discovery_traits.add(trait)
            continue

        raw_signal_ids = trait_signal_details.index.get_level_values(1).to_series(index=trait_signal_details.index).astype(str)
        raw_signal_ids_num = pd.to_numeric(raw_signal_ids, errors="coerce")

        if raw_signal_ids_num.notna().all():
            unique_signal_ids = sorted(raw_signal_ids_num.astype(int).unique().tolist())
            signal_map = {raw_id: mapped_id + 1 for mapped_id, raw_id in enumerate(unique_signal_ids)}
            signal_iter = [(raw_id, signal_map[raw_id]) for raw_id in unique_signal_ids]
        else:
            unique_signal_ids = sorted(raw_signal_ids.unique().tolist())
            signal_map = {raw_id: mapped_id + 1 for mapped_id, raw_id in enumerate(unique_signal_ids)}
            signal_iter = [(raw_id, signal_map[raw_id]) for raw_id in unique_signal_ids]

        per_cell_type_hits: dict[str, list[int]] = {cell_type: [] for cell_type in cell_types}
        for raw_signal_id, mapped_signal_id in signal_iter:
            try:
                signal_rows = trait_signal_details.xs(raw_signal_id, level=1, drop_level=False)
            except Exception:
                level_1 = trait_signal_details.index.get_level_values(1).astype(str)
                signal_rows = trait_signal_details.loc[level_1 == str(raw_signal_id)]
                if len(signal_rows) == 0:
                    continue

            cells_for_signal: list[str] = []
            for value in signal_rows[signal_cell_ids_col].astype(str):
                cells_for_signal.extend(_parse_cell_ids_csv(value).tolist())

            cells_for_signal = pd.Index(cells_for_signal, dtype=str).unique()
            cells_for_signal = adata.obs_names.intersection(cells_for_signal)
            if len(cells_for_signal) == 0:
                continue

            counts = adata.obs.loc[cells_for_signal, adata_biocol].astype(str).value_counts()
            for cell_type in cell_types:
                denominator = float(totals_by_type.get(cell_type, 0))
                if denominator <= 0:
                    continue

                fraction = float(counts.get(cell_type, 0)) / denominator
                if _passes_heatmap_inclusion(fraction, threshold):
                    per_cell_type_hits[cell_type].append(int(mapped_signal_id))

        for cell_type, hits in per_cell_type_hits.items():
            if hits:
                ann_map[trait][cell_type] = ",".join(map(str, sorted(set(hits))))

    return ann_map, no_discovery_traits


def _displayed_indep_signal_annotation(
    *,
    trait: str,
    cell_type: str,
    conditional_pass: bool,
    ann_map: dict[str, dict[str, str]],
    no_discovery_traits: set[str],
) -> str:
    """Return the exact independent-population label displayed in a heatmap cell."""
    annotation = ann_map.get(str(trait), {}).get(str(cell_type), "")
    if annotation:
        return annotation
    if str(trait) in no_discovery_traits and conditional_pass:
        return "1"
    return ""


def _build_heatmap_proportions_table(
    *,
    df_marginal: pd.DataFrame,
    df_intersect: pd.DataFrame,
    threshold: float,
    trait_dict: dict[str, str] | None = None,
    cell_type_dict: dict[str, str] | None = None,
    adata: AnnData | None = None,
    adata_biocol: str = "cell_ontology_class",
    ann_map: dict[str, dict[str, str]] | None = None,
    no_discovery_traits: set[str] | None = None,
) -> pd.DataFrame:
    """Create a tidy supplementary table in the exact row/column order of a heatmap."""
    trait_dict = trait_dict or {}
    cell_type_dict = cell_type_dict or {}
    ann_map = ann_map or {}
    no_discovery_traits = {str(trait) for trait in (no_discovery_traits or set())}

    if not df_marginal.index.equals(df_intersect.index):
        raise ValueError("Marginal and conditional heatmap tables must have identical trait order.")
    if not df_marginal.columns.equals(df_intersect.columns):
        raise ValueError("Marginal and conditional heatmap tables must have identical cell-type order.")

    traits = df_intersect.index.astype(str).tolist()
    cell_types = df_intersect.columns.astype(str).tolist()
    num_traits = len(traits)
    num_cell_types = len(cell_types)

    marginal_values = df_marginal.astype(float).to_numpy().reshape(-1)
    conditional_values = df_intersect.astype(float).to_numpy().reshape(-1)
    marginal_pass = np.asarray(
        [_passes_heatmap_inclusion(value, threshold) for value in marginal_values],
        dtype=bool,
    )
    conditional_pass = np.asarray(
        [_passes_heatmap_inclusion(value, threshold) for value in conditional_values],
        dtype=bool,
    )

    totals_by_type = _cell_type_count_map(adata, adata_biocol)
    repeated_traits = np.repeat(np.asarray(traits, dtype=object), num_cell_types)
    tiled_cell_types = np.tile(np.asarray(cell_types, dtype=object), num_traits)
    signal_annotations = [
        ann_map.get(str(trait), {}).get(str(cell_type), "")
        for trait, cell_type in zip(repeated_traits, tiled_cell_types)
    ]
    displayed_signal_annotations = [
        _displayed_indep_signal_annotation(
            trait=str(trait),
            cell_type=str(cell_type),
            conditional_pass=bool(passes),
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
        for trait, cell_type, passes in zip(
            repeated_traits, tiled_cell_types, conditional_pass
        )
    ]

    table = pd.DataFrame(
        {
            "trait_order": np.repeat(np.arange(1, num_traits + 1), num_cell_types),
            "cell_type_order": np.tile(np.arange(1, num_cell_types + 1), num_traits),
            "trait": repeated_traits,
            "trait_label": [trait_dict.get(trait, trait) for trait in repeated_traits],
            "cell_type": tiled_cell_types,
            "cell_type_label": [
                _strip_trailing_count(_display_cell_type_name(cell_type, cell_type_dict))
                for cell_type in tiled_cell_types
            ],
            "cell_type_total_cells": [int(totals_by_type.get(cell_type, 0)) for cell_type in tiled_cell_types],
            "marginal_cell_type_proportion": marginal_values,
            "marginal_passes_heatmap_inclusion": marginal_pass,
            "conditional_cell_type_proportion": conditional_values,
            "conditional_proportion_displayed": np.where(
                conditional_pass,
                conditional_values,
                0.0,
            ),
            "conditional_passes_heatmap_inclusion": conditional_pass,
            "independent_signals_passing_heatmap_inclusion": signal_annotations,
            "independent_signal_annotation_displayed": displayed_signal_annotations,
            "heatmap_inclusion_threshold": float(threshold),
            "heatmap_inclusion_rule": f"> {float(threshold):g}",
        }
    )
    return table


def _write_heatmap_proportions_csv(
    *,
    out_csv: str | Path,
    df_marginal: pd.DataFrame,
    df_intersect: pd.DataFrame,
    threshold: float,
    trait_dict: dict[str, str] | None = None,
    cell_type_dict: dict[str, str] | None = None,
    adata: AnnData | None = None,
    adata_biocol: str = "cell_ontology_class",
    ann_map: dict[str, dict[str, str]] | None = None,
    no_discovery_traits: set[str] | None = None,
) -> pd.DataFrame:
    """Write a manuscript-ready cell-type-proportion CSV and return its table."""
    table = _build_heatmap_proportions_table(
        df_marginal=df_marginal,
        df_intersect=df_intersect,
        threshold=threshold,
        trait_dict=trait_dict,
        cell_type_dict=cell_type_dict,
        adata=adata,
        adata_biocol=adata_biocol,
        ann_map=ann_map,
        no_discovery_traits=no_discovery_traits,
    )
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_csv, index=False)
    print(f"Saved heatmap cell-type proportions: {out_csv} ({len(table):,} rows)")
    return table


def _make_square_heatmap_figure(
    num_traits: int,
    num_cell_types: int,
    *,
    cell_size: float = 0.55,
    left_margin: float = 7.0,
    right_margin: float = 3.5,
    bottom_margin: float = 7.0,
    top_margin: float = 5.5,
):
    """
    Create a figure/axis where each unit heatmap cell is physically square.

    Margins are specified in inches, and the heatmap body is exactly
    num_cell_types × num_traits cells at cell_size inches per cell.
    """
    if num_traits <= 0 or num_cell_types <= 0:
        raise ValueError("Cannot plot an empty heatmap.")

    heatmap_width = num_cell_types * cell_size
    heatmap_height = num_traits * cell_size

    fig_width = left_margin + heatmap_width + right_margin
    fig_height = bottom_margin + heatmap_height + top_margin

    fig = plt.figure(figsize=(fig_width, fig_height))

    ax_left = left_margin / fig_width
    ax_bottom = bottom_margin / fig_height
    ax_width = heatmap_width / fig_width
    ax_height = heatmap_height / fig_height

    ax = fig.add_axes([ax_left, ax_bottom, ax_width, ax_height])
    ax.set_aspect("equal", adjustable="box")

    return fig, ax


def add_top_lines(
    ax,
    color_counts,
    names,
    *,
    y_axes=1.01,
    text_offset=0.02,
    linewidth=4,
    fontsize=32,
    fontweight="bold",
) -> None:
    total = max(1, sum(count for _, count in color_counts))
    start = 0
    for (color, count), name in zip(color_counts, names):
        count = int(count)
        end = start + count
        xmin, xmax = start / total, end / total
        if count > 0:
            ax.add_line(
                Line2D(
                    [xmin, xmax],
                    [y_axes, y_axes],
                    transform=ax.transAxes,
                    color=color,
                    linewidth=linewidth,
                    solid_capstyle="butt",
                    clip_on=False,
                )
            )
            ax.text(
                (xmin + xmax) / 2,
                y_axes + text_offset,
                name,
                ha="center",
                va="bottom",
                fontsize=fontsize,
                color=color,
                fontweight=fontweight,
                transform=ax.transAxes,
                clip_on=False,
            )
        start = end
    ax.figure.canvas.draw_idle()


def add_right_lines(
    ax,
    color_counts,
    names,
    *,
    x_axes=1.01,
    text_offset=0.02,
    linewidth=4,
    fontsize=32,
    fontweight="bold",
) -> None:
    total = max(1, sum(count for _, count in color_counts))
    start = 0
    for (color, count), name in zip(color_counts, names):
        count = int(count)
        end = start + count
        ymin, ymax = start / total, end / total
        if count > 0:
            ax.add_line(
                Line2D(
                    [x_axes, x_axes],
                    [ymin, ymax],
                    transform=ax.transAxes,
                    color=color,
                    linewidth=linewidth,
                    solid_capstyle="butt",
                    clip_on=False,
                )
            )
            ax.text(
                x_axes + text_offset,
                (ymin + ymax) / 2,
                name,
                ha="left",
                va="center",
                fontsize=fontsize,
                color=color,
                fontweight=fontweight,
                transform=ax.transAxes,
                clip_on=False,
                rotation=-90,
            )
        start = end
    ax.figure.canvas.draw_idle()


def add_dashed_diagonal_boxes(
    ax,
    trait_group_sizes: tuple[int, int, int],
    celltype_group_sizes: tuple[int, int, int],
    *,
    lw: float = 2.0,
    color: str = "black",
    linestyle: str = "--",
) -> None:
    """Draw Brain/Immune/Other diagonal guide boxes."""
    brain_t, immune_t, other_t = trait_group_sizes
    brain_c, immune_c, other_c = celltype_group_sizes

    x_starts = [0, brain_c, brain_c + immune_c]
    x_widths = [brain_c, immune_c, other_c]
    y_starts = [other_t + immune_t, other_t, 0]
    y_heights = [brain_t, immune_t, other_t]

    for x_start, x_width, y_start, y_height in zip(x_starts, x_widths, y_starts, y_heights):
        if x_width <= 0 or y_height <= 0:
            continue
        ax.add_patch(
            Rectangle(
                (x_start, y_start),
                x_width,
                y_height,
                fill=False,
                edgecolor=color,
                linestyle=linestyle,
                linewidth=lw,
                zorder=10,
            )
        )


def _add_heatmap_colorbar_and_legend(
    *,
    fig,
    ax,
    cmap,
    norm,
    boundaries,
    fontsize_mult: float,
    cbar_label: str = "Prop. sig. conditional cells",
) -> None:
    """Place colorbar and legend above the top group labels to avoid overlap."""
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    ax_pos = ax.get_position()
    cbar_left = ax_pos.x0
    cbar_bottom = min(0.94, ax_pos.y1 + 0.10)
    cbar_width = min(0.34, ax_pos.width * 0.55)
    cbar_height = 0.018

    cbar_ax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_height])
    cbar = fig.colorbar(
        sm,
        cax=cbar_ax,
        orientation="horizontal",
        boundaries=boundaries,
        ticks=[0, 0.5, 1.0],
        spacing="proportional",
        drawedges=True,
    )
    cbar.ax.set_xticklabels(["0%", "50%", "100%"], fontsize=20 * fontsize_mult)
    cbar.set_label(cbar_label, fontsize=24 * fontsize_mult, labelpad=12 * fontsize_mult)

    inferred_handle = Line2D(
        [],
        [],
        linestyle="None",
        marker="$1$",
        color="white",
        markersize=18 * fontsize_mult,
    )
    inferred_handle.set_path_effects([pe.withStroke(linewidth=2.5, foreground="black")])

    legend_elements = [
        Line2D(
            [],
            [],
            marker="*",
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="black",
            markeredgewidth=2,
            markersize=25 * fontsize_mult,
        ),
        Line2D(
            [],
            [],
            marker="*",
            linestyle="None",
            markerfacecolor="red",
            markeredgecolor="black",
            markersize=25 * fontsize_mult,
        ),
        inferred_handle,
    ]

    fig.legend(
        legend_elements,
        ["Marginal association", "Conditional association", "Inferred cell population"],
        loc="upper right",
        bbox_to_anchor=(0.985, 0.985),
        prop={"size": 22 * fontsize_mult},
        frameon=True,
        borderaxespad=0.5,
    )

## Curated subset heatmap

In [11]:
# Trait order for the curated subset heatmap, interpreted top-to-bottom.
DEFAULT_TRAIT_ORDER = subset_traits


def plot_combined_heatmap(
    df_marginal_props: pd.DataFrame,
    df_intersect_props: pd.DataFrame,
    *,
    title: str,
    cell_types: list[str],
    cell_type_dict: dict[str, str] | None = None,
    adata: AnnData | None = None,
    adata_biocol: str = "cell_ontology_class",
    df_signal_details: pd.DataFrame | None = None,
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
    trait_dict: dict[str, str] | None = None,
    trait_order: list[str] | None = None,
    fontsize_mult: float = 1.0,
    trait_group_sizes: tuple[int, int, int] = (6, 6, 8),
    celltype_group_sizes: tuple[int, int, int] = (8, 7, 9),
    threshold: float = 0.05,
    out_png: str | Path = "ct_level_fig.png",
    out_csv: str | Path | None = None,
) -> None:
    """Plot the curated heatmap with marginal and conditional discovery overlays."""
    trait_dict = trait_dict or {}
    cell_type_dict = cell_type_dict or {}
    trait_order = trait_order or DEFAULT_TRAIT_ORDER

    df_marginal_in = df_marginal_props.copy()
    df_intersect_in = df_intersect_props.copy()
    df_marginal_in.index = df_marginal_in.index.astype(str)
    df_intersect_in.index = df_intersect_in.index.astype(str)
    df_marginal_in.columns = df_marginal_in.columns.astype(str)
    df_intersect_in.columns = df_intersect_in.columns.astype(str)

    # Keep the curated subset only, then append any observed traits not in trait_order.
    available_traits = list(df_intersect_in.index.astype(str))
    ordered_traits = [trait for trait in trait_order if trait in set(available_traits)]
    remaining_traits = [trait for trait in available_traits if trait not in set(ordered_traits)]
    trait_index = ordered_traits + remaining_traits

    missing_traits = [trait for trait in trait_order if trait not in set(available_traits)]
    if missing_traits:
        print(f"[plot_combined_heatmap] Warning: {len(missing_traits)} traits were not found and were ignored.")

    raw_cell_types = [str(cell_type) for cell_type in cell_types]
    missing_cell_types = [
        cell_type
        for cell_type in raw_cell_types
        if cell_type not in df_marginal_in.columns or cell_type not in df_intersect_in.columns
    ]
    if missing_cell_types:
        print(f"[plot_combined_heatmap] Warning: {len(missing_cell_types)} cell types were not found and were ignored.")
        for cell_type in missing_cell_types:
            print(f"  - {cell_type}")

    raw_cell_types = [
        cell_type
        for cell_type in raw_cell_types
        if cell_type in df_marginal_in.columns and cell_type in df_intersect_in.columns
    ]

    if not raw_cell_types:
        raise ValueError("None of the requested curated cell types were available.")

    df_marginal = df_marginal_in.loc[trait_index, raw_cell_types].astype(float)
    df_intersect = df_intersect_in.loc[trait_index, raw_cell_types].astype(float)
    num_traits, num_cell_types = df_intersect.shape

    if df_signal_details is not None:
        if adata is None:
            raise ValueError("Provide adata when df_signal_details is provided.")
        ann_map, no_discovery_traits = _build_indep_sig_annotations(
            adata=adata,
            df_signal_details=df_signal_details,
            adata_biocol=adata_biocol,
            cell_types=raw_cell_types,
            threshold=threshold,
            trait_index=trait_index,
            signal_cell_ids_col=signal_cell_ids_col,
        )
    else:
        ann_map = {trait: {cell_type: "" for cell_type in raw_cell_types} for trait in trait_index}
        no_discovery_traits = set()

    fig, ax = _make_square_heatmap_figure(
        num_traits,
        num_cell_types,
        cell_size=0.55,
        left_margin=7.0,
        right_margin=3.5,
        bottom_margin=7.0,
        top_margin=5.5,
    )

    boundaries = np.linspace(0, 1, 12)
    cmap = ListedColormap(sns.color_palette("Blues", n_colors=11))
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    for trait_idx in range(num_traits):
        trait = str(df_intersect.index[trait_idx])
        for cell_type_idx, cell_type in enumerate(raw_cell_types):
            raw_intersect = float(df_intersect.iloc[trait_idx, cell_type_idx])
            raw_marginal = float(df_marginal.iloc[trait_idx, cell_type_idx])
            conditional_star = _passes_heatmap_inclusion(raw_intersect, threshold)
            marginal_star = _passes_heatmap_inclusion(raw_marginal, threshold)
            display_value = raw_intersect if conditional_star else 0.0

            x = cell_type_idx
            y = num_traits - trait_idx - 1
            facecolor = "white" if display_value == 0.0 else cmap(norm(display_value))
            ax.add_patch(Rectangle((x, y), 1, 1, facecolor=facecolor, edgecolor="none"))

            if marginal_star:
                ax.text(
                    x + 0.5,
                    y + 0.5,
                    "☆",
                    ha="center",
                    va="center",
                    fontsize=48 * fontsize_mult,
                    color="black",
                    fontweight="bold",
                    zorder=20,
                )

            if conditional_star:
                ax.text(
                    x + 0.5,
                    y + 0.5,
                    "★",
                    ha="center",
                    va="center",
                    fontsize=38 * fontsize_mult,
                    color="red",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")],
                    zorder=21,
                )

            annotation = ann_map.get(trait, {}).get(cell_type, "")
            if annotation:
                ax.text(
                    x + 0.95,
                    y + 0.95,
                    annotation,
                    ha="right",
                    va="top",
                    fontsize=16 * fontsize_mult,
                    color="white",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="black")],
                    zorder=30,
                )
            elif trait in no_discovery_traits and conditional_star:
                ax.text(
                    x + 0.95,
                    y + 0.95,
                    "1",
                    ha="right",
                    va="top",
                    fontsize=18 * fontsize_mult,
                    color="white",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="black")],
                    zorder=30,
                )

    ax.set_xlim(0, num_cell_types)
    ax.set_ylim(0, num_traits)
    ax.set_xticks(np.arange(num_cell_types) + 0.5)
    ax.set_yticks(np.arange(num_traits) + 0.5)
    ax.set_xticks(np.arange(num_cell_types + 1), minor=True)
    ax.set_yticks(np.arange(num_traits + 1), minor=True)
    ax.grid(which="minor", color="lightgray", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    x_labels = [
        _cell_type_label_with_count(
            cell_type,
            adata=adata,
            adata_biocol=adata_biocol,
            cell_type_dict=cell_type_dict,
        )
        for cell_type in raw_cell_types
    ]
    y_labels = list(reversed([trait_dict.get(trait, trait) for trait in df_intersect.index]))

    ax.set_xticklabels(x_labels, fontsize=24 * fontsize_mult, rotation=45, ha="right")
    ax.set_yticklabels(y_labels, fontsize=24 * fontsize_mult)

    brain_t, immune_t, other_t = trait_group_sizes
    brain_c, immune_c, other_c = celltype_group_sizes
    x_colors = (["red"] * brain_c) + (["blue"] * immune_c) + (["green"] * other_c)
    y_colors = (["green"] * other_t) + (["blue"] * immune_t) + (["red"] * brain_t)

    _color_ticklabels(ax.get_xticklabels(), x_colors, fontsize=24 * fontsize_mult, rotation=45, ha="right")
    _color_ticklabels(ax.get_yticklabels(), y_colors, fontsize=24 * fontsize_mult)

    add_top_lines(
        ax,
        color_counts=[("red", brain_c), ("blue", immune_c), ("green", other_c)],
        names=["Brain", "Immune", "Other"],
        fontsize=32 * fontsize_mult,
    )
    add_right_lines(
        ax,
        color_counts=[("green", other_t), ("blue", immune_t), ("red", brain_t)],
        names=["Other", "Immune", "Brain"],
        fontsize=32 * fontsize_mult,
    )
    add_dashed_diagonal_boxes(
        ax,
        trait_group_sizes=trait_group_sizes,
        celltype_group_sizes=celltype_group_sizes,
        lw=2.0,
        linestyle="--",
        color="black",
    )

    _add_heatmap_colorbar_and_legend(
        fig=fig,
        ax=ax,
        cmap=cmap,
        norm=norm,
        boundaries=boundaries,
        fontsize_mult=fontsize_mult,
    )

    if title:
        fig.suptitle(title, fontsize=30 * fontsize_mult)

    plt.savefig(out_png, bbox_inches="tight", dpi=300)
    if out_csv is not None:
        _write_heatmap_proportions_csv(
            out_csv=out_csv,
            df_marginal=df_marginal,
            df_intersect=df_intersect,
            threshold=threshold,
            trait_dict=trait_dict,
            cell_type_dict=cell_type_dict,
            adata=adata,
            adata_biocol=adata_biocol,
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
    plt.show()

In [12]:
subset_traits_available = [trait for trait in subset_traits if trait in df_marginal_props.index]

plot_combined_heatmap(
    df_marginal_props=df_marginal_props.loc[subset_traits_available],
    df_intersect_props=df_marginal_x_cond_props.loc[subset_traits_available],
    title="",
    cell_types=plot_cell_types,
    cell_type_dict=cell_type_dict,
    adata=adata,
    adata_biocol="cell_ontology_class",
    df_signal_details=df_signal_details,
    signal_cell_ids_col="marg_x_signal_cell_ids_cond_sig",
    trait_dict=trait_dict,
    trait_order=DEFAULT_TRAIT_ORDER,
    fontsize_mult=1.0,
    trait_group_sizes=(4, 4, 8),
    celltype_group_sizes=(8, 11, 13),
    threshold=HEATMAP_THRESHOLD,
    out_png="ct_level_fig.png",
    out_csv=CURATED_HEATMAP_PROPORTIONS_CSV,
)

Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/TMS_FACS_curated_heatmap_cell_type_proportions.csv (512 rows)


## All-trait/all-cell-type labels and group ordering

In [13]:
id_to_trait_name = {
    "PASS_ADHD_Demontis2018": "ADHD",
    "PASS_Alzheimers_Jansen2019": "Alzheimer's Disease",
    "PASS_AtrialFibrillation_Nielsen2018": "Atrial Fibrilation",
    "PASS_BIP_Mullins2021": "Bipolar Disorder",
    "PASS_CD_deLange2017": "Crohn's Disease",
    "PASS_Celiac": "Celiac Disease",
    "PASS_Coronary_Artery_Disease": "Coronary Artery Disease",
    "PASS_DrinksPerWeek_Liu2019": "Drinks Per Week",
    "PASS_FastingGlucose_Manning": "Fasting Glucose",
    "PASS_GeneralRiskTolerance_KarlssonLinner2019": "General Risk Tolerance",
    "PASS_IBD_deLange2017": "Inflammatory bowel disease",
    "PASS_Insomnia_Jansen2019": "Insomnia",
    "PASS_Intelligence_SavageJansen2018": "Intelligence",
    "PASS_Lupus": "Lupus",
    "PASS_MDD_Howard2019": "Major depressive disorder",
    "PASS_Multiple_sclerosis": "Multiple sclerosis",
    "PASS_Parkinsons23andMe_Corces2020": "Parkinson's Disease",
    "PASS_Primary_biliary_cirrhosis": "Primary biliary cirrhosis",
    "PASS_ReactionTime_Davies2018": "Reaction Time",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid arthritis",
    "PASS_SWB": "Subjective well-being",
    "PASS_Schizophrenia_Pardinas2018": "Schizophrenia",
    "PASS_SleepDuration_Dashti2019": "Sleep Duration",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes",
    "PASS_Type_2_Diabetes": "Type 2 Diabetes",
    "PASS_UC_deLange2017": "Ulcerative Colitis",
    "PASS_VerbalNumericReasoning_Davies2018": "Verbal Numeric Reasoning",
    "PASS_Worry_Nagel2018": "Worry",
    "UKB_460K.biochemistry_AlanineAminotransferase": "Alanine Aminotransferase",
    "UKB_460K.biochemistry_AlkalinePhosphatase": "Alkaline Phosphatase",
    "UKB_460K.biochemistry_Cholesterol": "Cholesterol",
    "UKB_460K.biochemistry_Glucose": "Glucose",
    "UKB_460K.biochemistry_HDLcholesterol": "HDLcholesterol",
    "UKB_460K.biochemistry_HbA1c": "HbA1c",
    "UKB_460K.biochemistry_LDLdirect": "LDLdirect",
    "UKB_460K.biochemistry_SHBG": "SHBG",
    "UKB_460K.biochemistry_Testosterone_Male": "Testosterone",
    "UKB_460K.biochemistry_TotalBilirubin": "TotalBilirubin",
    "UKB_460K.biochemistry_TotalProtein": "Total Protein",
    "UKB_460K.biochemistry_Triglycerides": "Triglycerides",
    "UKB_460K.blood_EOSINOPHIL_COUNT": "Eosinophil count",
    "UKB_460K.blood_LYMPHOCYTE_COUNT": "Lymphocyte count",
    "UKB_460K.blood_MEAN_CORPUSCULAR_HEMOGLOBIN": "Mean corpular hemoglobin",
    "UKB_460K.blood_MONOCYTE_COUNT": "Monocyte count",
    "UKB_460K.blood_PLATELET_COUNT": "Platelet count",
    "UKB_460K.blood_RBC_DISTRIB_WIDTH": "Red blood cell distribution width",
    "UKB_460K.blood_RED_COUNT": "Red blood cell count",
    "UKB_460K.blood_WHITE_COUNT": "White blood cell count",
    "UKB_460K.bmd_HEEL_TSCOREz": "Heel T-score",
    "UKB_460K.body_BALDING1": "Balding",
    "UKB_460K.body_BMIz": "BMI",
    "UKB_460K.body_HEIGHTz": "Height",
    "UKB_460K.body_WHRadjBMIz": "Waist-hip ratio adjusted for BMI",
    "UKB_460K.bp_DIASTOLICadjMEDz": "Diastolic Blood Pressure",
    "UKB_460K.bp_SYSTOLICadjMEDz": "Systolic Blood Pressure",
    "UKB_460K.cancer_BREAST": "Breast Cancer",
    "UKB_460K.cov_EDU_COLLEGE": "College Education",
    "UKB_460K.cov_EDU_YEARS": "Years of Education",
    "UKB_460K.cov_SMOKING_STATUS": "Smoking Status",
    "UKB_460K.disease_AID_ALL": "Auto immune traits",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED": "Eczema",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma",
    "UKB_460K.disease_CARDIOVASCULAR": "Cardiovascular Disease",
    "UKB_460K.disease_HYPERTENSION_DIAGNOSED": "Hypertension",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP": "Hypothyroidism",
    "UKB_460K.disease_RESPIRATORY_ENT": "Respiratory and Ear-nose-throat Diseases",
    "UKB_460K.impedance_BASAL_METABOLIC_RATEz": "Basal Metabolic Rate",
    "UKB_460K.lung_FEV1FVCzSMOKE": "FEV1-FVC Ratio",
    "UKB_460K.lung_FVCzSMOKE": "Forced Vital Capacity",
    "UKB_460K.mental_NEUROTICISM": "Neuroticism",
    "UKB_460K.other_MORNINGPERSON": "Morning Person",
    "UKB_460K.pigment_HAIR": "Hair Color",
    "UKB_460K.repro_MENARCHE_AGE": "Age at Menarche",
    "UKB_460K.repro_MENOPAUSE_AGE": "Age at Menopause",
    "UKB_460K.repro_NumberChildrenEverBorn_Pooled": "Number children ever born",
}

In [14]:
# =========================================================
# Biologically grouped ordering for the final all-trait/all-cell-type heatmap
#
# Bars and dashed boxes use three broad groups only:
#   Brain -> Immune -> Other
#
# Within "Other", blood traits/cell types are ordered first so the visual order is:
#   Brain -> Immune -> Blood -> all remaining Other
# =========================================================

BRAIN_CELL_TYPES = [
    "neuronal stem cell",
    "neuroepithelial cell",
    "ependymal cell",
    "astrocyte",
    "Bergmann glial cell",
    "oligodendrocyte precursor cell",
    "oligodendrocyte",
    "microglial cell",
    "brain pericyte",
    "interneuron",
    "medium spiny neuron",
    "neuron",
]

IMMUNE_CELL_TYPES = [
    "lymphoid progenitor cell",
    "thymocyte",
    "DN4 thymocyte",
    "epithelial cell of thymus",
    "T cell",
    "mature alpha-beta T cell",
    "CD4-positive, alpha-beta T cell",
    "regulatory T cell",
    "CD8-positive, alpha-beta T cell",
    "NK cell",
    "mature NK T cell",
    "B cell",
    "naive B cell",
    "immature B cell",
    "precursor B cell",
    "late pro-B cell",
    "early pro-B cell",
    "plasma cell",
    "lymphocyte",
    "leukocyte",
    "myeloid cell",
    "myeloid leukocyte",
    "monocyte",
    "classical monocyte",
    "intermediate monocyte",
    "non-classical monocyte",
    "promonocyte",
    "macrophage",
    "lung macrophage",
    "Kupffer cell",
    "myeloid dendritic cell",
    "plasmacytoid dendritic cell",
    "dendritic cell",
    "professional antigen presenting cell",
    "granulocyte",
    "neutrophil",
    "basophil",
]

BLOOD_CELL_TYPES = [
    "hematopoietic stem cell",
    "granulocyte monocyte progenitor cell",
    "megakaryocyte-erythroid progenitor cell",
    "proerythroblast",
    "granulocytopoietic cell",
]

OTHER_CELL_TYPES = [
    # Pancreatic / metabolic / liver / adipose
    "pancreatic A cell",
    "pancreatic B cell",
    "pancreatic D cell",
    "pancreatic PP cell",
    "pancreatic acinar cell",
    "pancreatic ductal cell",
    "pancreatic stellate cell",
    "hepatocyte",
    "endothelial cell of hepatic sinusoid",
    "mesenchymal stem cell of adipose",

    # Cardiovascular / vascular / muscle / stromal
    "endocardial cell",
    "valve cell",
    "atrial myocyte",
    "ventricular myocyte",
    "fibroblast of cardiac tissue",
    "endothelial cell of coronary artery",
    "aortic endothelial cell",
    "vein endothelial cell",
    "endothelial cell of lymphatic vessel",
    "endothelial cell",
    "fenestrated cell",
    "pericyte cell",
    "smooth muscle cell",
    "smooth muscle cell of the pulmonary artery",
    "bronchial smooth muscle cell",
    "smooth muscle cell of trachea",
    "fibroblast",
    "fibrocyte",
    "adventitial cell",
    "stromal cell",
    "mesenchymal stem cell",
    "skeletal muscle satellite cell",
    "chondrocyte",

    # Respiratory / airway
    "respiratory basal cell",
    "basal epithelial cell of tracheobronchial tree",
    "ciliated columnar cell of tracheobronchial tree",
    "club cell of bronchiole",
    "mucus secreting cell",
    "lung neuroendocrine cell",
    "type I pneumocyte",
    "type II pneumocyte",
    "fibroblast of lung",
    "pulmonary interstitial fibroblast",

    # Gut epithelium
    "intestinal crypt stem cell",
    "enteroendocrine cell",
    "enterocyte of epithelium of large intestine",
    "large intestine goblet cell",
    "Brush cell of epithelium proper of large intestine",
    "epithelial cell of large intestine",
    "secretory cell",
    "epithelial cell",

    # Skin / bladder / mammary epithelium
    "basal cell of epidermis",
    "keratinocyte stem cell",
    "bulge keratinocyte",
    "keratinocyte",
    "epidermal cell",
    "basal cell",
    "bladder urothelial cell",
    "bladder cell",
    "luminal epithelial cell of mammary gland",

    # Kidney
    "epithelial cell of proximal tubule",
    "kidney loop of Henle ascending limb epithelial cell",
    "kidney collecting duct epithelial cell",
    "kidney collecting duct principal cell",
    "mesangial cell",
    "kidney interstitial fibroblast",
]

CELL_TYPE_ORDER = BRAIN_CELL_TYPES + IMMUNE_CELL_TYPES + BLOOD_CELL_TYPES + OTHER_CELL_TYPES
CELL_TYPE_GROUPS = {
    "Brain": BRAIN_CELL_TYPES,
    "Immune": IMMUNE_CELL_TYPES,
    "Other": BLOOD_CELL_TYPES + OTHER_CELL_TYPES,
}


BRAIN_TRAITS = [
    # Neuropsychiatric / cognitive / behavior, plus BMI as a near-brain trait.
    "PASS_ADHD_Demontis2018",
    "PASS_BIP_Mullins2021",
    "PASS_Schizophrenia_Pardinas2018",
    "PASS_MDD_Howard2019",
    "UKB_460K.mental_NEUROTICISM",
    "PASS_Worry_Nagel2018",
    "PASS_Insomnia_Jansen2019",
    "PASS_SleepDuration_Dashti2019",
    "UKB_460K.other_MORNINGPERSON",
    "PASS_Alzheimers_Jansen2019",
    "PASS_Parkinsons23andMe_Corces2020",
    "PASS_Intelligence_SavageJansen2018",
    "PASS_VerbalNumericReasoning_Davies2018",
    "PASS_ReactionTime_Davies2018",
    "PASS_GeneralRiskTolerance_KarlssonLinner2019",
    "PASS_SWB",
    "PASS_DrinksPerWeek_Liu2019",
    "UKB_460K.cov_EDU_YEARS",
    "UKB_460K.cov_EDU_COLLEGE",
    "UKB_460K.body_BMIz",
]

IMMUNE_TRAITS = [
    "PASS_Multiple_sclerosis",
    "PASS_Rheumatoid_Arthritis",
    "PASS_Lupus",
    "PASS_Celiac",
    "PASS_IBD_deLange2017",
    "PASS_UC_deLange2017",
    "PASS_CD_deLange2017",
    "PASS_Type_1_Diabetes",
    "PASS_Primary_biliary_cirrhosis",
    "UKB_460K.disease_AID_ALL",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "UKB_460K.disease_ASTHMA_DIAGNOSED",
]

BLOOD_TRAITS = [
    "UKB_460K.blood_RED_COUNT",
    "UKB_460K.blood_RBC_DISTRIB_WIDTH",
    "UKB_460K.blood_MEAN_CORPUSCULAR_HEMOGLOBIN",
    "UKB_460K.blood_PLATELET_COUNT",
    "UKB_460K.blood_WHITE_COUNT",
    "UKB_460K.blood_LYMPHOCYTE_COUNT",
    "UKB_460K.blood_MONOCYTE_COUNT",
    "UKB_460K.blood_EOSINOPHIL_COUNT",
]

OTHER_TRAITS = [
    # Metabolic / endocrine / pancreatic / hepatic
    "PASS_Type_2_Diabetes",
    "PASS_FastingGlucose_Manning",
    "UKB_460K.biochemistry_Glucose",
    "UKB_460K.biochemistry_HbA1c",
    "UKB_460K.body_WHRadjBMIz",
    "UKB_460K.impedance_BASAL_METABOLIC_RATEz",
    "UKB_460K.biochemistry_Testosterone_Male",
    "UKB_460K.biochemistry_SHBG",
    "UKB_460K.repro_MENARCHE_AGE",
    "UKB_460K.repro_MENOPAUSE_AGE",
    "UKB_460K.repro_NumberChildrenEverBorn_Pooled",
    "UKB_460K.biochemistry_AlanineAminotransferase",
    "UKB_460K.biochemistry_AlkalinePhosphatase",
    "UKB_460K.biochemistry_TotalBilirubin",
    "UKB_460K.biochemistry_TotalProtein",

    # Cardiovascular / lipids / vascular
    "PASS_AtrialFibrillation_Nielsen2018",
    "PASS_Coronary_Artery_Disease",
    "UKB_460K.disease_CARDIOVASCULAR",
    "UKB_460K.disease_HYPERTENSION_DIAGNOSED",
    "UKB_460K.bp_SYSTOLICadjMEDz",
    "UKB_460K.bp_DIASTOLICadjMEDz",
    "UKB_460K.biochemistry_Cholesterol",
    "UKB_460K.biochemistry_HDLcholesterol",
    "UKB_460K.biochemistry_LDLdirect",
    "UKB_460K.biochemistry_Triglycerides",

    # Respiratory / airway and other
    "UKB_460K.disease_RESPIRATORY_ENT",
    "UKB_460K.lung_FEV1FVCzSMOKE",
    "UKB_460K.lung_FVCzSMOKE",
    "UKB_460K.cov_SMOKING_STATUS",
    "UKB_460K.cancer_BREAST",
    "UKB_460K.body_HEIGHTz",
    "UKB_460K.bmd_HEEL_TSCOREz",
    "UKB_460K.body_BALDING1",
    "UKB_460K.pigment_HAIR",
]

TRAIT_ORDER = BRAIN_TRAITS + IMMUNE_TRAITS + BLOOD_TRAITS + OTHER_TRAITS
TRAIT_GROUPS = {
    "Brain": BRAIN_TRAITS,
    "Immune": IMMUNE_TRAITS,
    "Other": BLOOD_TRAITS + OTHER_TRAITS,
}

## Final all-trait/all-cell-type heatmap

In [15]:
def plot_combined_heatmap_reordered(
    df_marginal_props: pd.DataFrame,
    df_intersect_props: pd.DataFrame,
    *,
    title: str = "",
    trait_dict: dict[str, str] | None = None,
    trait_order: list[str] | None = None,
    cell_type_order: list[str] | None = None,
    trait_groups: dict[str, list[str]] | None = None,
    cell_type_groups: dict[str, list[str]] | None = None,
    adata: AnnData | None = None,
    adata_biocol: str = "cell_ontology_class",
    df_signal_details: pd.DataFrame | None = None,
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
    threshold: float = 0.05,
    fontsize_mult: float = 1.25,
    out_png: str | Path = "ct_level_fig_reordered.png",
    out_csv: str | Path | None = None,
) -> None:
    """
    Plot all observed traits and cell types.

    The final plot uses three annotation groups: Brain, Immune, and Other.
    Blood traits/cell types are ordered first within Other but are not drawn as
    a separate annotation-bar or dashed-box category.
    """
    trait_dict = trait_dict or {}

    df_marginal = df_marginal_props.copy()
    df_intersect = df_intersect_props.copy()
    df_marginal.index = df_marginal.index.astype(str)
    df_marginal.columns = df_marginal.columns.astype(str)
    df_intersect.index = df_intersect.index.astype(str)
    df_intersect.columns = df_intersect.columns.astype(str)

    if trait_groups is not None:
        ordered_traits, grouped_traits = _ordered_by_named_groups(
            df_intersect.index.tolist(),
            trait_groups,
            fallback_group="Other",
        )
    else:
        ordered_traits = _ordered_with_remainder(df_intersect.index.tolist(), trait_order or [])
        grouped_traits = {"Brain": [], "Immune": [], "Other": ordered_traits}

    if cell_type_groups is not None:
        ordered_cell_types, grouped_cell_types = _ordered_by_named_groups(
            df_intersect.columns.tolist(),
            cell_type_groups,
            fallback_group="Other",
        )
    else:
        ordered_cell_types = _ordered_with_remainder(df_intersect.columns.tolist(), cell_type_order or [])
        grouped_cell_types = {"Brain": [], "Immune": [], "Other": ordered_cell_types}

    # Ensure all three groups exist even if a caller passed a partial grouping dict.
    for group_name in ["Brain", "Immune", "Other"]:
        grouped_traits.setdefault(group_name, [])
        grouped_cell_types.setdefault(group_name, [])

    missing_traits_in_marginal = [trait for trait in ordered_traits if trait not in df_marginal.index]
    if missing_traits_in_marginal:
        raise ValueError(f"Traits missing from df_marginal_props: {missing_traits_in_marginal}")

    missing_cell_types_in_marginal = [cell_type for cell_type in ordered_cell_types if cell_type not in df_marginal.columns]
    if missing_cell_types_in_marginal:
        raise ValueError(f"Cell types missing from df_marginal_props: {missing_cell_types_in_marginal}")

    df_marginal = df_marginal.loc[ordered_traits, ordered_cell_types].astype(float)
    df_intersect = df_intersect.loc[ordered_traits, ordered_cell_types].astype(float)
    num_traits, num_cell_types = df_intersect.shape

    if df_signal_details is not None:
        if adata is None:
            raise ValueError("Provide adata when df_signal_details is provided.")
        ann_map, no_discovery_traits = _build_indep_sig_annotations(
            adata=adata,
            df_signal_details=df_signal_details,
            adata_biocol=adata_biocol,
            cell_types=ordered_cell_types,
            threshold=threshold,
            trait_index=ordered_traits,
            signal_cell_ids_col=signal_cell_ids_col,
        )
    else:
        ann_map = {trait: {cell_type: "" for cell_type in ordered_cell_types} for trait in ordered_traits}
        no_discovery_traits = set()

    fig, ax = _make_square_heatmap_figure(
        num_traits,
        num_cell_types,
        cell_size=0.30,
        left_margin=8.5,
        right_margin=4.0,
        bottom_margin=8.0,
        top_margin=6.0,
    )

    boundaries = np.linspace(0, 1, 12)
    cmap = ListedColormap(sns.color_palette("Blues", n_colors=11))
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    for trait_idx in range(num_traits):
        trait = str(df_intersect.index[trait_idx])
        for cell_type_idx, cell_type in enumerate(ordered_cell_types):
            raw_intersect = float(df_intersect.iloc[trait_idx, cell_type_idx])
            raw_marginal = float(df_marginal.iloc[trait_idx, cell_type_idx])
            conditional_star = _passes_heatmap_inclusion(raw_intersect, threshold)
            marginal_star = _passes_heatmap_inclusion(raw_marginal, threshold)
            display_value = raw_intersect if conditional_star else 0.0

            x = cell_type_idx
            y = num_traits - trait_idx - 1
            facecolor = "white" if display_value == 0.0 else cmap(norm(display_value))
            ax.add_patch(Rectangle((x, y), 1, 1, facecolor=facecolor, edgecolor="none"))

            if marginal_star:
                ax.text(
                    x + 0.5,
                    y + 0.5,
                    "☆",
                    ha="center",
                    va="center",
                    fontsize=24 * fontsize_mult,
                    color="black",
                    fontweight="bold",
                    zorder=20,
                )

            if conditional_star:
                ax.text(
                    x + 0.5,
                    y + 0.5,
                    "★",
                    ha="center",
                    va="center",
                    fontsize=19 * fontsize_mult,
                    color="red",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=1, foreground="black")],
                    zorder=21,
                )

            annotation = ann_map.get(trait, {}).get(cell_type, "")
            if annotation:
                ax.text(
                    x + 0.95,
                    y + 0.95,
                    annotation,
                    ha="right",
                    va="top",
                    fontsize=12 * fontsize_mult,
                    color="white",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="black")],
                    zorder=30,
                )
            elif trait in no_discovery_traits and conditional_star:
                ax.text(
                    x + 0.95,
                    y + 0.95,
                    "1",
                    ha="right",
                    va="top",
                    fontsize=13 * fontsize_mult,
                    color="white",
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="black")],
                    zorder=30,
                )

    ax.set_xlim(0, num_cell_types)
    ax.set_ylim(0, num_traits)
    ax.set_xticks(np.arange(num_cell_types) + 0.5)
    ax.set_yticks(np.arange(num_traits) + 0.5)
    ax.set_xticks(np.arange(num_cell_types + 1), minor=True)
    ax.set_yticks(np.arange(num_traits + 1), minor=True)
    ax.grid(which="minor", color="lightgray", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    x_labels = [
        _cell_type_label_with_count(
            cell_type,
            adata=adata,
            adata_biocol=adata_biocol,
            cell_type_dict=None,
        )
        for cell_type in ordered_cell_types
    ]
    y_labels = list(reversed([trait_dict.get(trait, trait) for trait in df_intersect.index]))

    ax.set_xticklabels(x_labels, fontsize=15 * fontsize_mult, rotation=45, ha="right")
    ax.set_yticklabels(y_labels, fontsize=15 * fontsize_mult)

    brain_t = len(grouped_traits["Brain"])
    immune_t = len(grouped_traits["Immune"])
    other_t = len(grouped_traits["Other"])

    brain_c = len(grouped_cell_types["Brain"])
    immune_c = len(grouped_cell_types["Immune"])
    other_c = len(grouped_cell_types["Other"])

    x_colors = (["red"] * brain_c) + (["blue"] * immune_c) + (["green"] * other_c)
    y_colors = (["green"] * other_t) + (["blue"] * immune_t) + (["red"] * brain_t)

    _color_ticklabels(ax.get_xticklabels(), x_colors, fontsize=15 * fontsize_mult, rotation=45, ha="right")
    _color_ticklabels(ax.get_yticklabels(), y_colors, fontsize=15 * fontsize_mult)

    add_top_lines(
        ax,
        color_counts=[("red", brain_c), ("blue", immune_c), ("green", other_c)],
        names=["Brain", "Immune", "Other"],
        fontsize=22 * fontsize_mult,
    )
    add_right_lines(
        ax,
        color_counts=[("green", other_t), ("blue", immune_t), ("red", brain_t)],
        names=["Other", "Immune", "Brain"],
        fontsize=22 * fontsize_mult,
    )
    add_dashed_diagonal_boxes(
        ax,
        trait_group_sizes=(brain_t, immune_t, other_t),
        celltype_group_sizes=(brain_c, immune_c, other_c),
        lw=2.0,
        linestyle="--",
        color="black",
    )

    _add_heatmap_colorbar_and_legend(
        fig=fig,
        ax=ax,
        cmap=cmap,
        norm=norm,
        boundaries=boundaries,
        fontsize_mult=fontsize_mult,
    )

    if title:
        fig.suptitle(title, fontsize=30 * fontsize_mult)

    plt.savefig(out_png, bbox_inches="tight", dpi=300)
    if out_csv is not None:
        _write_heatmap_proportions_csv(
            out_csv=out_csv,
            df_marginal=df_marginal,
            df_intersect=df_intersect,
            threshold=threshold,
            trait_dict=trait_dict,
            cell_type_dict=None,
            adata=adata,
            adata_biocol=adata_biocol,
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
    plt.show()

In [16]:
plot_combined_heatmap_reordered(
    df_marginal_props=df_marginal_props,
    df_intersect_props=df_marginal_x_cond_props,
    title="",
    trait_dict=id_to_trait_name,
    trait_order=TRAIT_ORDER,
    cell_type_order=CELL_TYPE_ORDER,
    trait_groups=TRAIT_GROUPS,
    cell_type_groups=CELL_TYPE_GROUPS,
    adata=adata,
    adata_biocol="cell_ontology_class",
    df_signal_details=df_signal_details,
    signal_cell_ids_col="marg_x_signal_cell_ids_cond_sig",
    threshold=HEATMAP_THRESHOLD,
    fontsize_mult=1.0,
    out_png="ct_level_fig_all.png",
    out_csv=ALL_TRAIT_HEATMAP_PROPORTIONS_CSV,
)

Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/TMS_FACS_all_trait_heatmap_cell_type_proportions.csv (8,700 rows)
